In [8]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'pihps_food_inflation_indonesia.csv'

print('Project root:', PROJECT_ROOT)
print('Dataset path :', DATA_PATH)

Project root: C:\Users\caran\Desktop\WORK\Projects\Misc\Mercatorie
Dataset path : C:\Users\caran\Desktop\WORK\Projects\Misc\Mercatorie\data\raw\pihps_food_inflation_indonesia.csv


# Core libraries
- **pandas, numpy** for data handling
- **matplotlib, seaborn** for exploration
- **scikit-learn** for preprocessing, pipelines, training, and evaluation
- **xgboost** for a strong tabular-data baseline
- **shap** for model explainability
- **joblib** for saving trained artifacts
- **streamlit** for the dashboard later

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

print('Starter imports loaded successfully.')

Starter imports loaded successfully.


# Load Dataset

In [10]:
# load data
df = pd.read_csv(DATA_PATH)

# dataset overview
print("Shape:", df.shape)
display(df.head())

# column names
print("Columns:")
for col in df.columns:
    print("-", col)

# data types
display(df.dtypes.to_frame("dtype"))

# date coverage
df["date"] = pd.to_datetime(df["date"])
print("Date range:", df["date"].min(), "to", df["date"].max())

# missing values
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0])

# unique commodities
print("Number of commodities:", df["commodity"].nunique())
print("Commodity list:", sorted(df["commodity"].unique()))

# variable groups by type
categorical_cols = ["commodity"]
binary_or_flag_cols = ["is_observed_source", "is_month_start", "is_month_end"]
time_cols = ["date", "year", "month", "quarter", "day_of_week", "week_of_year"]
target_cols = ["target_7d_inflation_pct", "target_30d_inflation_pct"]

numeric_feature_cols = [
    col for col in df.columns
    if col not in categorical_cols + binary_or_flag_cols + time_cols + target_cols
]

print("\nCategorical columns:", categorical_cols)
print("Binary/flag columns:", binary_or_flag_cols)
print("Time columns:", time_cols)
print("Target columns:", target_cols)
print("Numeric feature columns:", numeric_feature_cols)

print("\nCounts")
print("Total columns:", len(df.columns))
print("Categorical:", len(categorical_cols))
print("Binary/flag:", len(binary_or_flag_cols))
print("Time-related:", len(time_cols))
print("Targets:", len(target_cols))
print("Numeric features:", len(numeric_feature_cols))

# target distributions
display(df[target_cols].describe().T)

# primary & secondary modeling target for this project
primary_target = "target_7d_inflation_pct"
secondary_target = "target_30d_inflation_pct"
print("\nPrimary target selected for modeling:", primary_target)
print("\nPSecondary target selected for modeling:", secondary_target)

Shape: (12320, 25)


,date,commodity,commodity_code,is_observed_source,price_idr,year,month,quarter,day_of_week,week_of_year,is_month_start,is_month_end,price_lag_1d,price_lag_7d,price_lag_30d,price_change_1d_pct,price_change_7d_pct,price_change_30d_pct,rolling_mean_7d,rolling_mean_30d,rolling_std_30d,price_vs_ma30_pct,volatility_30d_pct,target_7d_inflation_pct,target_30d_inflation_pct
0,2018-01-31,beras,1,1,12100.0,2018,1,1,2,5,0,1,12100.0,12100.0,11050.0,0.0,0.0,9.5023,12100.0,11965.0000,127.4078,1.1283,1.1522,0.0,-0.8264
1,2018-02-01,beras,1,1,12100.0,2018,2,1,3,5,1,0,12100.0,12100.0,11750.0,0.0,0.0,2.9787,12100.0,11976.6667,122.9896,1.0298,0.1808,0.0,-0.8264
2,2018-02-02,beras,1,1,12100.0,2018,2,1,4,5,0,0,12100.0,12100.0,11750.0,0.0,0.0,2.9787,12100.0,11988.3333,117.2114,0.9315,0.1808,0.0,-0.8264
3,2018-02-03,beras,1,0,12100.0,2018,2,1,5,5,0,0,12100.0,12100.0,11750.0,0.0,0.0,2.9787,12100.0,12000.0000,109.8588,0.8333,0.1808,0.0,-0.8264
4,2018-02-04,beras,1,0,12100.0,2018,2,1,6,5,0,0,12100.0,12100.0,11800.0,0.0,0.0,2.5424,12100.0,12010.0000,104.5516,0.7494,0.1706,0.0,-1.2397


Columns:
- date
- commodity
- commodity_code
- is_observed_source
- price_idr
- year
- month
- quarter
- day_of_week
- week_of_year
- is_month_start
- is_month_end
- price_lag_1d
- price_lag_7d
- price_lag_30d
- price_change_1d_pct
- price_change_7d_pct
- price_change_30d_pct
- rolling_mean_7d
- rolling_mean_30d
- rolling_std_30d
- price_vs_ma30_pct
- volatility_30d_pct
- target_7d_inflation_pct
- target_30d_inflation_pct


,dtype
date,object
commodity,object
commodity_code,int64
is_observed_source,int64
price_idr,float64
year,int64
month,int64
quarter,int64
day_of_week,int64
week_of_year,int64


Date range: 2018-01-31 00:00:00 to 2024-10-29 00:00:00


Series([], dtype: int64)

Number of commodities: 5
Commodity list: ['bawang_merah', 'beras', 'cabai_merah', 'minyak_goreng', 'telur_ayam']

Categorical columns: ['commodity']
Binary/flag columns: ['is_observed_source', 'is_month_start', 'is_month_end']
Time columns: ['date', 'year', 'month', 'quarter', 'day_of_week', 'week_of_year']
Target columns: ['target_7d_inflation_pct', 'target_30d_inflation_pct']
Numeric feature columns: ['commodity_code', 'price_idr', 'price_lag_1d', 'price_lag_7d', 'price_lag_30d', 'price_change_1d_pct', 'price_change_7d_pct', 'price_change_30d_pct', 'rolling_mean_7d', 'rolling_mean_30d', 'rolling_std_30d', 'price_vs_ma30_pct', 'volatility_30d_pct']

Counts
Total columns: 25
Categorical: 1
Binary/flag: 3
Time-related: 6
Targets: 2
Numeric features: 13


,count,mean,std,min,25%,50%,75%,max
target_7d_inflation_pct,12320.0,0.250980,6.558166,-51.5306,-0.900900,0.0,0.9524,115.0000
target_30d_inflation_pct,12320.0,0.914218,11.794271,-60.9455,-2.632675,0.0,3.2198,167.3684



Primary target selected for modeling: target_7d_inflation_pct

PSecondary target selected for modeling: target_30d_inflation_pct
